In [1]:
import pandas as pd
import torch
import numpy as np
import torch.nn as nn
#from sklearn.preprocessing import StandardScaler
#from sklearn.model_selection import LeaveOneGroupOut
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
#from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import copy
import random
from sklearn.model_selection import train_test_split
import polars as pl
from pathlib import Path
import torch.profiler
import torch.nn.functional as F
from enum import Enum

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
class ModelType(Enum):
    CNN1D = 1
    CNN_LSTM = 2


In [3]:
data_path = '../../data/'
training_data_path = data_path + "Final Training Data/"

In [4]:
class CNN1D(nn.Module):
    def __init__(self, input_channels, num_classes, activation_fn):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 32, 5, padding=2),
            nn.BatchNorm1d(32),
            activation_fn(),

            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            activation_fn(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128),
            activation_fn(),
            nn.MaxPool1d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            activation_fn(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.mean(dim=2)
        return self.fc(x)

In [5]:
class CNN_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(CNN_LSTM, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #cnn takes input of shape (batch_size, channels, seq_len)
        x = x.permute(0, 2, 1)
        out = self.cnn(x)
        # lstm takes input of shape (batch_size, seq_len, input_size)
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [7]:
def get_loader(settings, X_train, y_train):
    batch_size = 0
    sampler = None
    shuffle = True
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = 1.0 / class_counts
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        batch_size = 65536
    elif settings["weight"] == "random_weighted":

        loss_fn = torch.nn.CrossEntropyLoss()
        class_counts = Counter(y_train.numpy())
        
        num_samples = len(y_train)

        class_weights = {cls: num_samples / count for cls, count in class_counts.items()}

        sample_weights = [class_weights[label.item()] for label in y_train]
        sample_weights = torch.DoubleTensor(sample_weights)

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        batch_size = 3000
        shuffle = None
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])
        batch_size = settings["batch size"]

    return DataLoader(
                TensorDataset(X_train, y_train),
                batch_size=batch_size,
                shuffle=shuffle,
                sampler=sampler,
                num_workers=2,
                pin_memory=True,
                prefetch_factor=2
            ), loss_fn

In [8]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch, y_batch) in enumerate(loader):
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

In [9]:
def validate_epoch(model, X_val, y_val):
    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    with torch.no_grad():

        X_val_device = X_val.to(device)
        y_val_device = y_val.to(device)

        logits = model(X_val_device)

        probs = torch.softmax(logits, dim=1)

        confidences, preds = torch.max(probs, dim=1)

        mask = confidences >= 0.8

        if mask.sum() == 0:

            val_metric = 0

        else:

            coverage = (
                mask.float().mean().item()
            )

            precision = (
                preds[mask] == y_val_device[mask]
            ).float().mean().item()

            val_metric = (
                precision * coverage
            )

    return val_metric

In [30]:
def train_loso_ensemble(
    X_train,
    X_test,
    y_train,
    y_test,
    subject_idx,
    settings,
):
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.long, device=device)

    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.long, device=device)

    # ----------------------------
    # validation split
    # ----------------------------
    X_train_np = X_train.cpu().numpy()
    y_train_np = y_train.cpu().numpy()

    X_train_main, X_val, y_train_main, y_val = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.2,
        stratify=y_train_np
    )

    X_train_main = torch.tensor(X_train_main, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)

    y_train_main = torch.tensor(y_train_main, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # ----------------------------
    # ensemble
    # ----------------------------
    ensamble_settings = settings["training settings"]["ensamble settings"]
    model_settings = settings["model settings"]
    models_with_scores = []

    for ensemble_idx in range(ensamble_settings["number of trained models"]):

        print(ensemble_idx + 1)
        seed = subject_idx * 1000 + ensemble_idx
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        lr_range = ensamble_settings["learning rate range"]
        wd_range = ensamble_settings["weight decay range"]

        match model_settings["model"]:
            case ModelType.CNN1D:
                model = CNN1D(
                    input_channels=X_train.shape[-1],
                    num_classes=len(torch.unique(y_train)),
                    activation_fn=model_settings["activation_fn"]
                ).to(device)
            case ModelType.CNN_LSTM:
                model = CNN_LSTM(
                    X_train.shape[-1],
                    64,
                    1,
                    len(torch.unique(y_train))
                ).to(device)
        model = torch.compile(model)
        
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=np.random.uniform(lr_range[0], lr_range[1]),
            weight_decay=np.random.uniform(wd_range[0], wd_range[1])
        )

        loader, loss_fn = get_loader(ensamble_settings, X_train_main, y_train_main)

        best_val = -1
        best_state = model.state_dict()
        patience_counter = 0


        
        for epoch in range(ensamble_settings["epoch"]):
            #with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CUDA, torch.profiler.ProfilerActivity.CPU]) as prof:
            train_epoch(model, loader, optimizer, loss_fn)
                
            #print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
            val_score = validate_epoch(model, X_val, y_val)

            if val_score > best_val:
                best_val = val_score
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= ensamble_settings["patience"]:
                break
        model.load_state_dict(best_state)
        models_with_scores.append((best_val, model))

    # ----------------------------
    # TEST
    # ----------------------------
    models_with_scores.sort(key=lambda x: x[0], reverse=True)
    top_models = [m for _, m in models_with_scores[:min(ensamble_settings["number of trained models"], ensamble_settings["number of used models"])]]

    with torch.no_grad():

        probs_list = []
        for m in top_models:
            m.eval()
            prob = torch.softmax(m(X_test), dim=1).cpu()
            probs_list.append(prob)

        mean_probs = torch.mean(torch.stack(probs_list), dim=0)

        confidences, preds = torch.max(mean_probs, dim=1)

        confidences = confidences.numpy()
        preds = preds.cpu()
        y_test_np = y_test.cpu()

        results = []

        for t in settings["training settings"]["thresholds"]:
            mask = confidences >= t

            if mask.sum() == 0:
                results.append((t, 0, 0.0))
                continue

            precision = (preds[mask] == y_test_np[mask]).float().mean().item()

            results.append((t, mask.sum().item(), precision))

        acc = (preds == y_test_np).float().mean().item()

    return acc, results, len(preds)

In [11]:
def load_subject(cache_dir, subject):
    data = np.load(Path(cache_dir) / f"{subject}.npz", allow_pickle=True)
    return data["X"], data["y"]

def get_subjects(cache_dir):
    return sorted([p.stem for p in Path(cache_dir).glob("*.npz")])

In [12]:
def normalize_train_test(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

In [32]:
results_log = []
settings = {
    "training settings": {
        "thresholds": [0.7, 0.8, 0.9, 0.95],
        "ensamble settings": {
            "learning rate range": [0.0008, 0.0020],
            "weight decay range": [0.0002, 0.0035],
        
            "number of trained models": 8,
            "number of used models": 3,
            "patience": 5,
            "epoch": 150,
        
            "weight": None,
            "label smoothing": 0.08,
            "batch size": 256
        }
    },
    "model settings": {
        "model": ModelType.CNN_LSTM,
        "activation_fn": nn.GELU
    }
}


cache_dir = "../data/Final Training Data/Windowed Data/fa58a95392aed78461eb668748d46ab8"

subjects = get_subjects(cache_dir)

subject_idx = 0
for test_subject in subjects:
    subject_idx += 1

    train_subjects = [s for s in subjects if s != test_subject]

    print(f"\nSubject {subject_idx}: {test_subject}")
    print("Loading data...")

    # -------------------
    # LOAD TRAIN
    # -------------------
    X_train_list, y_train_list = [], []

    for s in train_subjects:
        X_s, y_s = load_subject(cache_dir, s)
        X_train_list.append(X_s)
        y_train_list.append(y_s)

    X_train = np.concatenate(X_train_list)
    y_train = np.concatenate(y_train_list)

    # -------------------
    # LOAD TEST
    # -------------------
    X_test, y_test = load_subject(cache_dir, test_subject)

    # -------------------
    # NORMALIZATION)
    # -------------------
    X_train, X_test = normalize_train_test(X_train, X_test)

    print("Training...")

    acc, results, predict_count = train_loso_ensemble(
        X_train, X_test,
        y_train, y_test,
        subject_idx, settings
    )
    
    results_log.append({
        "subject": test_subject,
        "acc": acc,
        "results": results,
        "predict_count": predict_count
    })
    
    print(f"acc: {acc}")
    for i in range(len(results)):
        result = results[i]
        print(f"confidence threshold (%): {result[0] * 100}, coverage: {result[1]}/{predict_count}, of which correct (%) : {result[2] * 100:.4f}")

    print("\n")


Subject 1: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1
acc: 0.5240253806114197
confidence threshold (%): 70.0, coverage: 68/1103, of which correct (%) : 41.1765
confidence threshold (%): 80.0, coverage: 1/1103, of which correct (%) : 0.0000
confidence threshold (%): 90.0, coverage: 0/1103, of which correct (%) : 0.0000
confidence threshold (%): 95.0, coverage: 0/1103, of which correct (%) : 0.0000



Subject 2: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
1
acc: 0.4264870882034302
confidence threshold (%): 70.0, coverage: 0/1782, of which correct (%) : 0.0000
confidence threshold (%): 80.0, coverage: 0/1782, of which correct (%) : 0.0000
confidence threshold (%): 90.0, coverage: 0/1782, of which correct (%) : 0.0000
confidence threshold (%): 95.0, coverage: 0/1782, of which correct (%) : 0.0000



Subject 3: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
1
acc: 0.6212435364723206
confidence threshold (%): 70.0, coverage: 406

In [28]:
mean_acc = sum(r["acc"] for r in results_log) / len(results_log)
print(f"mean acc: {mean_acc}")

num_thresholds = len(results_log[0]["results"])
for i in range(num_thresholds):
    threshold = results_log[0]["results"][i][0]

    mean_coverage = (
        sum(
            r["results"][i][1] / r["predict_count"]
            for r in results_log
        )
        / len(results_log)
    )
    
    mean_precision = (
        sum(r["results"][i][2] for r in results_log)
        / len(results_log)
    )

    print(
        f"threshold: {threshold * 100:.2f} %, "
        f"mean coverage: {mean_coverage*100:.2f} % "
        f"mean precision: {mean_precision*100:.4f}"
    )

print("\n")

for i, log in enumerate(results_log):
    print(f"Subject number {i+1}: {log.get('subject')}")
    print(f"acc: {log.get('acc')}")
    for result in log.get("results"):
        print(f"conf threshold (%): {result[0] * 100}, coverage: {result[1]}/{log.get('predict_count')}, correct (%) : {result[2] * 100:.4f}")

    print("\n")

ZeroDivisionError: division by zero

In [ ]:
del train_df, test_df, X_train, y_train, X_test, y_test

In [ ]:
print(f"mean test acc: {np.mean(test_acc_arr)}")
for i in range(len(train_acc_arr) - 1):
    curr_conf = test_min_conf_pass_count_arr[i]
    print(f"train acc: {train_acc_arr[i]:.4f}, test acc: {test_acc_arr[i]:.4f}, test average confidence: {test_conf_arr[i]:.4f} \n")
    
    for j in range(len(curr_conf)):
        print(f"threshold: {curr_conf[j][0]}, coverage: {curr_conf[j][1]}, precision: {curr_conf[j][2]:.4f}")

    print("\n")

In [ ]:
folds = np.arange(1, len(train_acc) + 1)
print(test_acc)
plt.figure(figsize=(10, 5))

train_acc_proc = np.array(train_acc) * 100
test_acc_proc = np.array(test_acc) * 100
test_conf_arr_proc = np.array(test_conf_arr) * 100

plt.plot(folds, train_acc_proc, marker='o', label="Train accuracy")
plt.plot(folds, test_acc_proc, marker='o', label="Test accuracy")
plt.plot(folds, test_conf_arr_proc, marker='o', label="Avg confidence")

plt.xlabel("LOSO fold")
plt.ylabel("Score %")
plt.xticks(folds)
plt.legend()
plt.tight_layout()
plt.show()

mean_coverages = []
mean_precisions = []

for threshold_idx in range(len(thresholds)):
    
    coverages = []
    precisions = []

    for fold in test_min_conf_pass_count_arr:
        
        coverage = fold[threshold_idx][1]
        precision = fold[threshold_idx][2]

        passed, total = map(int, coverage.split("/"))
        coverage_ratio = passed / total

        coverages.append(coverage_ratio * 100)
        precisions.append(precision * 100)

    mean_coverages.append(np.mean(coverages))
    mean_precisions.append(np.mean(precisions))

plt.figure(figsize=(10, 5))

plt.plot(thresholds, mean_coverages, marker='o', label="Coverage")
plt.plot(thresholds, mean_precisions, marker='o', label="Precision")

plt.hlines(mean_coverages, thresholds[0], thresholds, linestyles="dashed", alpha=0.3)
plt.hlines(mean_precisions, thresholds[0], thresholds, linestyles="dashed", alpha=0.3)

plt.xlabel("Confidence threshold %")
plt.ylabel("Score %")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.unique(y, return_counts=True)